# Projet d'Optimisation et Machinne learning

In [226]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Membres: Charles Pilleur, Ancco Jhonatan

#  Classificateur multiclasse de la base de données UCI HAR:

https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones

Les données utilisées proviennent d’expériences menées avec un groupe de 30 volontaires, âgés de 19 à 48 ans.
Chaque participant a exécuté six activités différentes : WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING et LAYING, tout en portant un smartphone Samsung Galaxy S II fixé à la taille.

Chaque enregistrement du jeu de données contient :

l’accélération triaxiale totale et l’accélération corporelle estimée,

la vitesse angulaire triaxiale issue du gyroscope,

un vecteur de 561 caractéristiques,

l’étiquette d’activité correspondante,

et l’identifiant du sujet ayant effectué l’expérience.

Le but final est d’utiliser ces caractéristiques pour entraîner un modèle de classification supervisée capable de reconnaître automatiquement l’activité physique réalisée par un individu à partir des signaux des capteurs du smartphone.

In [227]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

X_train = np.loadtxt('/content/drive/MyDrive/DataProjetOpti/train/X_train.txt')
y_train = np.loadtxt('/content/drive/MyDrive/DataProjetOpti/train/y_train.txt')
X_test = np.loadtxt('/content/drive/MyDrive/DataProjetOpti/test/X_test.txt')
y_test = np.loadtxt('/content/drive/MyDrive/DataProjetOpti/test/y_test.txt')

print(f"Training shape: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test shape: {X_test.shape}, Labels: {y_test.shape}")

Training shape: (7352, 561), Labels: (7352,)
Test shape: (2947, 561), Labels: (2947,)


On va prendre seulement 22 caracteristiques pour tester notre optimisation.

In [228]:
selected_features = list(range(0, 561))

# Select only these features
X_train_selected = (X_train[:, selected_features])[:2000]
X_test_selected = (X_test[:, selected_features])[:2000]

print(f"Training shape: {X_train_selected.shape}, Labels: {y_train.shape}")
print(f"Test shape: {X_test_selected.shape}, Labels: {y_test.shape}")

Training shape: (2000, 561), Labels: (7352,)
Test shape: (2000, 561), Labels: (2947,)


# Fonction à optimiser
$$
\min_{x\in\mathbb{R}^{(M+1)K}}  g(x) + \lambda h(x),
$$
où
$$
h(x) = \sum_{\ell=1}^{L}
\log\Big(1 + \sum_{k\ne\ell}
\exp\big[\mu_\ell + \varphi(u_\ell)^\top(x^{(k)} - x^{(z_l)})\big]\Big),
$$
$$
x^{(k)} = [w^{(k)}; b^{(k)}] \in \mathbb{R}^{M+1},
$$
$$
g(x) = \sum_{k=1}^{K} \|w^{(k)}\|_1, \,
$$
avec :
- **μ_ℓ = 1** : Marge fixée à 1 pour tous les échantillons
- **φ(u_ℓ) = [u_ℓ^⊤, 1]^⊤** ∈ ℝ^{M+1} : Transformation qui ajoute un biais de 1
- **x^{(k)}** ∈ ℝ^{M+1} : Paramètres pour la classe k (vecteur de poids + biais)
- **(u_ℓ, z_ℓ)** : ℓ-ème échantillon d'entraînement (vecteur, classe)

---


In [229]:
def phi(u):
    if u.ndim == 1:
        return np.concatenate([u, [1.0]])
    else:
        # Cas où u est une matrice (L, N)
        L, N = u.shape
        return np.hstack([u, np.ones((L, 1))])

def extract_weights_biases(x, N, K):
    # Remodeler x en matrice (K, N+1)
    x_mat = x.reshape(K, N+1)

    # Extraire les poids (premières N colonnes) et les biais (dernière colonne)
    w = x_mat[:, :-1]  # forme (K, N)
    b = x_mat[:, -1]   # forme (K,)

    return w, b, x_mat
def h_loss(x, X_train, y_train, mu=1.0):
    L, N = X_train.shape
    K = len(np.unique(y_train))

    w, b, x_mat = extract_weights_biases(x, N, K)

    scores = X_train @ w.T + b.reshape(1, -1)  # (L, K)

    total_loss = 0.0

    for i in range(L):
        true_class = int(y_train[i]) - 1  # convertir en index 0-based

        # Score pour la classe vraie
        score_true = scores[i, true_class]

        # Initialiser la somme des exponentielles pour les classes incorrectes
        sum_exp = 0.0

        # Pour chaque classe incorrecte
        for k in range(K):
            if k != true_class:
                # Score pour la classe k
                score_k = scores[i, k]
                # Terme à l'intérieur de l'exponentielle
                exp_term = mu + (score_k - score_true)
                sum_exp += np.exp(exp_term)

        # Ajouter la perte pour cet échantillon
        total_loss += np.log(1.0 + sum_exp)

    return total_loss

def g_reg(x, N, K):
    # Extraire les poids (sans les biais)
    w, _, _ = extract_weights_biases(x, N, K)

    # Calculer la norme L1 : somme des valeurs absolues
    return np.sum(np.abs(w))

def objective_function(x, X_train, y_train, lambda_reg, mu=1.0):
    L, N = X_train.shape
    K = len(np.unique(y_train))

    loss = h_loss(x, X_train, y_train, mu)

    reg = g_reg(x, N, K)

    return reg + lambda_reg * loss

**Formulation pour l'algorithme de Condat**

L'algorithme de Condat résout des problèmes de la forme :
$$
\min_x f(x) + g(x)
$$

Pour notre problème de classification multiclasse avec régularisation ℓ₁, nous utilisons la **perte logistique** comme fonction différentiable et la **norme ℓ₁** comme régularisation non-différentiable. Cela correspond au cas particulier où **h = 0** :

$$
\min_x \underbrace{\lambda \sum_{\ell=1}^{L} \log\Big(1 + \sum_{k \neq z_\ell} \exp\big[1 + \varphi(u_\ell)^\top(x^{(k)} - x^{(z_\ell)})\big]\Big)}_{f(x)} + \underbrace{\sum_{k=1}^K \|w^{(k)}\|_1}_{g(x)}
$$

Dans cette formulation :
- **f(x)=Λ*h(x)** est la perte logistique multipliée par λ, fonction différentiable
- **g(x)** est la norme ℓ₁ sur les poids, fonction non-différentiable mais proximable





**Initialisation :**
- $x_0 \in \mathbb{R}^{(M+1)K}$
- $\tau \in (0, \frac{2}{L})$ où $L$ est la constante de Lipschitz de $\nabla f$

**Pour $n = 0, 1, 2, \dots$ :**

**Étape 1 : Calcul du gradient**
$$
\nabla h(x_n) =\sum_{\ell=1}^{L} \nabla h_\ell(x_n)
$$




In [230]:
def h_loss_gradient_vectorized(x, X_train, y_train, mu=1.0):
    """
    Version vectorisée du gradient de la perte avec marge
    """
    L, M = X_train.shape
    K = len(np.unique(y_train))

    # Reformater x en matrice (M+1, K)
    x_mat = x.reshape(K, M + 1).T  # (M+1, K)

    # Construire la matrice Phi: chaque colonne est φ(u_i)
    # φ(u_i) = [u_i; 1]
    Phi = np.vstack([X_train.T, np.ones((1, L))])  # (M+1, L)

    # Initialiser le gradient
    grad_mat = np.zeros((M + 1, K))

    # Convertir y_train en indices 0-based
    y_idx = y_train.astype(int) - 1

    # Pour chaque échantillon (peut être vectorisé)
    for i in range(L):
        phi_i = Phi[:, i:i+1]  # (M+1, 1)
        true_class = y_idx[i]

        # Calculer tous les scores s_k = φ_i^T x^{(k)}
        scores = phi_i.T @ x_mat  # (1, K)

        # Extraire le score de la classe vraie
        s_true = scores[0, true_class]

        # Calculer a_k pour toutes les classes
        # a_k = exp(mu + s_k - s_true) pour k ≠ true_class
        a = np.exp(mu + scores - s_true)  # (1, K)
        a[0, true_class] = 0  # Mettre à 0 pour la classe vraie

        # Calculer S = 1 + Σ a_k
        S = 1.0 + np.sum(a)

        # Calculer les probabilités
        p_true = 1.0 / S
        p_other = a / S  # (1, K)

        # Calculer la contribution au gradient pour cet échantillon
        # Pour la classe vraie: -φ_i * (1 - p_true)
        grad_mat[:, true_class] -= phi_i.flatten() * (1.0 - p_true)

        # Pour les autres classes: φ_i * p_k
        for k in range(K):
            if k != true_class:
                grad_mat[:, k] += phi_i.flatten() * p_other[0, k]

    # Retourner le gradient sous forme de vecteur
    return grad_mat.T.flatten()

**Étape 2 : Descente de gradient**
$$
Z_n = x_n - \tau \nabla f(x_n)
$$

Ici $\tau<2/Lip(f)$.

In [231]:
def descende(x, X_train, y_train, tau, mu=1.0):

    # ∇f(x)
    grad_f =  lambda_reg*h_loss_gradient_vectorized(x, X_train, y_train, mu)

    # Descente de gradient: new_x = x - τ * ∇h(x)
    new_x = x - tau * grad_f

    return new_x

def estimate_lipschitz_constant(X_train, y_train, lambda_reg):
    L_samples, M = X_train.shape
    K = len(np.unique(y_train))

    # Compute the matrix Phi (with bias)
    Phi = np.hstack([X_train, np.ones((L_samples, 1))])

    # Compute the maximum eigenvalue of Phi^T Phi
    eig_max = np.linalg.eigvalsh(Phi.T @ Phi).max()

    # Bound for the Lipschitz constant of the gradient of h
    L_h_bound = 0.5 * eig_max

    # Then for f = λ * h, the Lipschitz constant of ∇f is λ * L_h_bound
    L_bound = lambda_reg * L_h_bound

    return L_bound

print("tau<",2/estimate_lipschitz_constant(X_train, y_train, lambda_reg))

tau< 2.0685172158721704e-06


**Étape 3 : Application de l'opérateur proximal (soft-thresholding)**
Pour chaque classe $k = 1, \dots, K$ :
- Extraire $Z_n^{(k)} = [w_n^{(k)}; b_n^{(k)}] \in \mathbb{R}^{M+1}$
- Appliquer le seuillage doux sur les poids :
  $$
  w_{n+1,i}^{(k)} = \text{sign}(w_{n,i}^{(k)}) \max(|w_{n,i}^{(k)}| - \tau, 0), \quad i = 1, \dots, M
  $$
- Conserver le biais inchangé :
  $$
  b_{n+1}^{(k)} = b_n^{(k)}
  $$
- Reforme :
  $$
  x_{n+1}^{(k)} = [w_{n+1}^{(k)}; b_{n+1}^{(k)}]
  $$


In [232]:
def prox_soft_thresholding(Z_n, M, tau):
    # Nombre de classes
    K = 6

    # Initialiser le vecteur résultat
    x_new = np.zeros_like(Z_n)

    # Pour chaque classe k = 0, 1, ..., K-1
    for k in range(K):
        # Indices pour la classe k
        start_idx = k * (M + 1)          # Début du bloc de la classe k
        end_idx_w = start_idx + M        # Fin des poids, début du biais
        idx_b = end_idx_w                # Indice du biais

        # Extraire les poids et le biais de Z_n
        w_k = Z_n[start_idx:end_idx_w]   # Poids pour la classe k (taille M)
        b_k = Z_n[idx_b]                 # Biais pour la classe k

        # Appliquer soft-thresholding sur les poids
        # w_new = sign(w) * max(|w| - τ, 0)
        w_new = np.sign(w_k) * np.maximum(np.abs(w_k) - tau, 0)

        # Conserver le biais inchangé
        b_new = b_k

        # Reforme dans x_new
        x_new[start_idx:end_idx_w] = w_new
        x_new[idx_b] = b_new

    return x_new

In [233]:
def forward_backward_algorithm(x0, X_train_selected, y_train, lambda_reg, tau,
                               mu=1.0, max_iter=1000, tol=1e-6, verbose=True):
    """
    Algorithme Forward-Backward complet utilisant tes fonctions

    Problème: min_x f(x) + g(x) où f(x) = lambda_reg * h_loss(x) et g(x) =  ||w||_1
    """
    L, N = X_train_selected.shape
    K = len(np.unique(y_train))

    # Initialisation
    x = x0.copy()

    # Vérification de la taille
    if len(x) != (N + 1) * K:
        raise ValueError(f"Taille incorrecte: x a {len(x)} éléments, mais attendu {(N+1)*K}")

    # Valeur initiale de l'objectif
    if verbose:
        f_init = h_loss(x, X_train_selected, y_train, mu)
        g_init = g_reg(x, N, K)
        obj_init = f_init + lambda_reg * g_init
        print(f"Début optimisation:")
        print(f"  f(x0) = {f_init:.4f}, g(x0) = {g_init:.4f}, objectif = {obj_init:.4f}")
        print(f"  Dimensions: {L} échantillons, {N} features, {K} classes")

    # Boucle d'optimisation
    for iteration in range(max_iter):
        # Sauvegarder x pour critère d'arrêt
        x_old = x.copy()

        # ===== ÉTAPE 1: FORWARD - Descente de gradient =====
        # Z_n = x_n - τ * ∇f(x_n)
        Z = descende(x, X_train_selected, y_train, tau=tau, mu=mu)

        x = prox_soft_thresholding(Z, N, tau * lambda_reg)

        # ===== Calcul de l'objectif pour suivi =====
        if verbose and iteration % 100 == 0:
            h_val = h_loss(x, X_train_selected, y_train, mu)
            g_val = g_reg(x, N, K)
            obj_val = lambda_reg *h_val +  g_val

            print(f"Iter {iteration:4d}: h={h_val:.4f}, g={g_val:.4f}, "
                  f"obj={obj_val:.4f}")

        # ===== Critère d'arrêt =====
        if np.linalg.norm(x - x_old) < tol:
            if verbose:
                print(f"\nConvergence atteinte à l'itération {iteration+1}")
            break

    # Valeurs finales
    if verbose:
        f_final = lambda_reg * h_loss(x, X_train_selected, y_train, mu)
        g_final = g_reg(x, N, K)
        obj_final = f_final + g_final

        print(f"\nRésultats finaux:")
        print(f"  f(x) = {f_final:.4f}")
        print(f"  g(x) = {g_final:.4f}")
        print(f"  Objectif = {obj_final:.4f}")

        # Analyse des poids
        w_final, b_final, _ = extract_weights_biases(x, N, K)
        print(f"\nAnalyse des poids:")
        for k in range(K):
            non_zero = np.sum(np.abs(w_final[k, :]) > 1e-6)
            print(f"  Classe {k+1}: {non_zero}/{N} poids non-nuls ({non_zero/N*100:.1f}%)")

    return x

In [248]:
def initialize_parameters_simple(X_train, y_train, method='zeros'):
    L, N = X_train.shape
    K = len(np.unique(y_train))

    if method == 'zeros':
        x0 = np.zeros((N + 1) * K)

    elif method == 'random':
        x0 = np.random.randn((N + 1) * K) * 0.01

    elif method == 'log_freq':
        # Initialisation à zéro pour les poids
        x0 = np.zeros((N + 1) * K)

        # Calcul des fréquences des classes
        counts = np.bincount(y_train.astype(int) - 1)
        frequencies = counts / np.sum(counts)
        log_freq = np.log(frequencies + 1e-9)  # + epsilon pour éviter log(0)

        # Initialiser les biais avec les log-fréquences
        for k in range(K):
            idx_b = k * (N + 1) + N  # indice du biais pour la classe k
            x0[idx_b] = log_freq[k]

    else:
        raise ValueError(f"Méthode d'initialisation inconnue: {method}")

    return x0

In [253]:
print("Dimensions des données:")
print(f"X_train_selected: {X_train_selected.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test_selected: {X_test_selected.shape}")

L, N = X_train_selected.shape
K = len(np.unique(y_train))
print(f"\nProblème: {L} échantillons, {N} features, {K} classes")
print(f"Taille du vecteur x: (N+1)*K = {N+1}*{K} = {(N+1)*K}")

lambda_reg = 1
tau = 2e-05          # pas d'apprentissage
mu = 1.0            # marge

print(f"\nHyperparamètres fixes:")
print(f"  λ  = {lambda_reg}")
print(f"  τ (pas) = {tau}")
print(f"  μ (marge) = {mu}")

x0 = initialize_parameters_simple(X_train_selected, y_train, method='log_freq')
x0 = x_optimal
print(f"\nInitialisation: vecteur x0 de taille {len(x0)}")

obj_init = lambda_reg * h_loss(x0, X_train_selected, y_train, mu) + g_reg(x0, N, K)
print(f"Valeur objectif initiale: {obj_init:.4f}")

print("\n" + "="*60)
print("DÉBUT DE L'OPTIMISATION FORWARD-BACKWARD")
print("="*60)

x_optimal = forward_backward_algorithm(
    x0, X_train_selected, y_train,
    lambda_reg=lambda_reg,
    tau=tau,
    mu=mu,
    max_iter=1000,
    tol=1e-6,
    verbose=True
)

obj_final = lambda_reg * h_loss(x_optimal, X_train_selected, y_train, mu) + g_reg(x_optimal, N, K)
print(f"\n" + "="*60)
print("RÉSUMÉ")
print("="*60)
print(f"Objectif initial: {obj_init:.4f}")
print(f"Objectif final:   {obj_final:.4f}")
print(f"Amélioration:     {((obj_init - obj_final)/obj_init*100):.1f}%")

Dimensions des données:
X_train_selected: (2000, 561)
y_train: (7352,)
X_test_selected: (2000, 561)

Problème: 2000 échantillons, 561 features, 6 classes
Taille du vecteur x: (N+1)*K = 562*6 = 3372

Hyperparamètres fixes:
  λ  = 1
  τ (pas) = 2e-05
  μ (marge) = 1.0

Initialisation: vecteur x0 de taille 3372
Valeur objectif initiale: 521.5361

DÉBUT DE L'OPTIMISATION FORWARD-BACKWARD
Début optimisation:
  f(x0) = 328.0783, g(x0) = 193.4578, objectif = 521.5361
  Dimensions: 2000 échantillons, 561 features, 6 classes
Iter    0: h=328.0043, g=193.4689, obj=521.4732
Iter  100: h=320.8884, g=194.5130, obj=515.4014
Iter  200: h=314.2827, g=195.4565, obj=509.7393
Iter  300: h=308.1337, g=196.3110, obj=504.4447
Iter  400: h=302.3919, g=197.0781, obj=499.4701
Iter  500: h=297.0129, g=197.7789, obj=494.7918
Iter  600: h=291.9586, g=198.4329, obj=490.3915
Iter  700: h=287.2003, g=199.0376, obj=486.2379
Iter  800: h=282.7124, g=199.5926, obj=482.3050
Iter  900: h=278.4731, g=200.1032, obj=478.576

In [254]:
def predict(x, X_data, N, K, unique_labels):
    # Extraire les poids et les biais
    w, b, _ = extract_weights_biases(x, N, K)

    # Calculer les scores pour chaque classe
    # scores shape: (n_samples, K)
    scores = X_data @ w.T + b.reshape(1, -1)

    # Obtenir l'indice de la classe avec le score le plus élevé
    # predicted_indices shape: (n_samples,)
    predicted_indices = np.argmax(scores, axis=1)

    # Convertir les indices 0-based en étiquettes originales
    predicted_labels = np.array([unique_labels[idx] for idx in predicted_indices])

    return predicted_labels

unique_labels = np.unique(y_train)

# Make predictions on the selected test data using the model trained on the small dataset
y_pred = predict(x_optimal, X_test_selected, N, K, unique_labels)

# Get the true labels for the selected test set
y_true = y_test[:len(X_test_selected)]

# Calculate accuracy
accuracy= np.mean(y_pred == y_true)

print(f"Accuracy on reduced test set (using x_optimal_small): {accuracy:.4f}")

# You can also print a classification report or confusion matrix for more details
from sklearn.metrics import classification_report, confusion_matrix

print("\nClassification Report (Reduced Test Set):")
print(classification_report(y_true, y_pred, zero_division=0))

print("\nConfusion Matrix (Reduced Test Set):")
print(confusion_matrix(y_true, y_pred))


Accuracy on reduced test set (using x_optimal_small): 0.8975

Classification Report (Reduced Test Set):
              precision    recall  f1-score   support

         1.0       0.95      0.91      0.93       360
         2.0       0.96      0.91      0.93       313
         3.0       0.84      0.94      0.89       276
         4.0       0.85      0.76      0.81       330
         5.0       0.79      0.88      0.83       357
         6.0       1.00      0.98      0.99       364

    accuracy                           0.90      2000
   macro avg       0.90      0.90      0.90      2000
weighted avg       0.90      0.90      0.90      2000


Confusion Matrix (Reduced Test Set):
[[328   1  31   0   0   0]
 [  9 284  20   0   0   0]
 [  7   9 260   0   0   0]
 [  0   2   0 252  76   0]
 [  0   0   0  43 314   0]
 [  0   0   0   0   7 357]]


On introduit une variable auxiliaire $z$ pour découpler les deux termes :

$$
\begin{aligned}
\min_{x,z} & \quad \lambda h(x) + g(z) \\
\text{s.t.} & \quad x - z = 0
\end{aligned}
$$

où :
- $x$ porte la perte logistique (différentiable)
- $z$ porte la régularisation L1 (non-différentiable)

### **Lagrangien Augmenté :**
$$
\mathcal{L}_\rho(x, z, u) = \lambda h(x) + g(z) + u^T(x - z) + \frac{\rho}{2}\|x - z\|_2^2
$$

avec :
- $u \in \mathbb{R}^{(M+1)K}$ : multiplicateur de Lagrange
- $\rho > 0$ : paramètre de pénalité

## **Algorithme ADMM - Étapes Itératives**

### **Initialisation :**
$$
x^0, z^0, u^0 \in \mathbb{R}^{(M+1)K} \quad \text{(souvent nuls)}, \quad \rho > 0
$$

### **Itération $t = 0, 1, 2, \dots$ :**

#### **Étape 1 : Mise à jour de $x$**
$$
x^{t+1} = \arg\min_{x} \left[ \lambda h(x) + \frac{\rho}{2}\|x - z^t + u^t\|_2^2 \right]
$$
C'est un problème lisse qu'on résout par **descente de gradient** :
$$
\nabla_x \mathcal{L} = \lambda \nabla h(x) + \rho(x - z^t + u^t)
$$

#### **Étape 2 : Mise àjour de $z$**
$$
z^{t+1} = \arg\min_{z} \left[ g(z) + \frac{\rho}{2}\|x^{t+1} - z + u^t\|_2^2 \right]
$$

Pour $g(z) = \|w_z\|_1$, solution fermée par **soft-thresholding** :

Pour chaque classe $k$ et chaque composante $i$ :
$$
w_{z,i}^{(k)} = \mathcal{S}_{\frac{1}{\rho}}\left(x_{w,i}^{(k)} + u_{w,i}^{(k)}\right)
$$
où $\mathcal{S}_\kappa(a) = \operatorname{sign}(a) \max(|a| - \kappa, 0)$

Les biais restent inchangés :
$$
b_z^{(k)} = x_b^{(k)} + u_b^{(k)}
$$

#### **Étape 3 : Mise à jour du multiplicateur**
$$
u^{t+1} = u^t + (x^{t+1} - z^{t+1})
$$

## **4. Conditions de Convergence**

### **Critères d'Arrêt :**

#### **Résidu Primal :**
$$
r^{t+1} = x^{t+1} - z^{t+1}
$$
Mesure la violation de contrainte.

#### **Résidu Dual :**
$$
s^{t+1} = \rho(z^{t+1} - z^t)
$$
Mesure le changement dans les variables duales.

#### **Condition de Convergence :**
L'algorithme s'arrête quand :
$$
\|r^{t+1}\|_2 \leq \epsilon^{\text{primal}} \quad \text{et} \quad \|s^{t+1}\|_2 \leq \epsilon^{\text{dual}}
$$
avec $\epsilon^{\text{primal}} = \epsilon^{\text{abs}} + \epsilon^{\text{rel}}\max(\|x^t\|_2, \|z^t\|_2)$


In [ ]:
import numpy as np
from scipy import sparse

In [256]:
def admm_solver(X_train, y_train, lambda_reg, rho=1.0, max_iter=100, tol=1e-4):
    """
    Résout le problème d'optimisation avec ADMM :
    min_x g(x) + λ*h(x) où g(x) = ||w||_1
    Reformulation : min_{x,z} λ*h(x) + g(z) s.t. x = z
    """
    L, N = X_train.shape
    K = len(np.unique(y_train))
    n_vars = (N + 1) * K

    # Initialisation
    x = x_admm
    z = np.zeros(n_vars)
    u = np.zeros(n_vars)

    # Précalculer certaines matrices pour l'étape x
    Phi = np.hstack([X_train, np.ones((L, 1))])  # Ajout du biais

    for it in range(max_iter):
        z_old = z.copy()
        # ===== Étape 1 : Mise à jour de x =====
        # x = argmin_x [λ*h(x) + (ρ/2)||x - z + u||^2]
        # On utilise un pas de gradient pour cette étape
        grad_x = lambda_reg * h_loss_gradient_vectorized(x, X_train, y_train, mu=1.0) + rho * (x - z + u)
        alpha = 1e-4  # Pas d'apprentissage pour la sous-étape x
        x = x - alpha * grad_x

        # ===== Étape 2 : Mise à jour de z =====
        # z = argmin_z [g(z) + (ρ/2)||x - z + u||^2]
        # g(z) = ||w||_1 -> soft-thresholding
        v = x + u
        z_new = np.zeros_like(z)

        for k in range(K):
            start_idx = k * (N + 1)
            end_idx_w = start_idx + N
            idx_b = end_idx_w

            # Poids : soft-thresholding
            w_k = v[start_idx:end_idx_w]
            w_new = np.sign(w_k) * np.maximum(np.abs(w_k) - 1/rho, 0)

            # Biais : pas de régularisation
            b_new = v[idx_b]

            z_new[start_idx:end_idx_w] = w_new
            z_new[idx_b] = b_new

        z = z_new

        # ===== Étape 3 : Mise à jour du multiplicateur =====
        u = u + x - z

        # ===== Vérification de la convergence =====
        primal_res = np.linalg.norm(x - z)
        dual_res = rho * np.linalg.norm(z - z_old if 'z_old' in locals() else z)

        if it % 10 == 0:
            obj_val = lambda_reg * h_loss(x, X_train, y_train) + g_reg(x, N, K)
            print(f"ADMM Iter {it:4d}: Obj = {obj_val:.4f}, Primal = {primal_res:.4f}")

        if primal_res < tol and dual_res < tol:
            print(f"ADMM convergé à l'itération {it}")
            break

    return x

# Application de ADMM
print("\n" + "="*60)
print("APPLICATION DE LA MÉTHODE ADMM")
print("="*60)

# Paramètres ADMM
lambda_reg_admm = 1.0
rho_admm = 1.0  # Paramètre de pénalité

x_admm = admm_solver(
    X_train_selected,
    y_train[:2000],  # Prendre les mêmes labels que X_train_selected
    lambda_reg_admm,
    rho=rho_admm,
    max_iter=200
)

# Évaluation du modèle ADMM
y_pred_admm = predict(x_admm, X_test_selected, N, K, unique_labels)
accuracy_admm = np.mean(y_pred_admm == y_true)

print(f"\nAccuracy avec ADMM: {accuracy_admm:.4f}")
print("\nClassification Report (ADMM):")
print(classification_report(y_true, y_pred_admm, zero_division=0))
print("\nConfusion Matrix (ADMM):")
print(confusion_matrix(y_true, y_pred_admm))

# Comparaison avec Forward-Backward
print("\n" + "="*60)
print("COMPARAISON DES MÉTHODES")
print("="*60)
print(f"Forward-Backward Accuracy: {accuracy:.4f}")
print(f"ADMM Accuracy: {accuracy_admm:.4f}")


APPLICATION DE LA MÉTHODE ADMM
ADMM Iter    0: Obj = 2506.5288, Primal = 7.8013
ADMM Iter   10: Obj = 2248.6017, Primal = 2.1106
ADMM Iter   20: Obj = 1882.5140, Primal = 0.9849
ADMM Iter   30: Obj = 1098.1419, Primal = 0.6134
ADMM Iter   40: Obj = 844.1947, Primal = 0.4514
ADMM Iter   50: Obj = 816.0600, Primal = 0.3634
ADMM Iter   60: Obj = 791.8493, Primal = 0.3126
ADMM Iter   70: Obj = 769.3247, Primal = 0.2488
ADMM Iter   80: Obj = 748.2625, Primal = 0.2038
ADMM Iter   90: Obj = 728.5735, Primal = 0.1670
ADMM Iter  100: Obj = 710.7658, Primal = 0.1479
ADMM Iter  110: Obj = 721.2733, Primal = 0.1163
ADMM Iter  120: Obj = 3759.2158, Primal = 0.3157
ADMM Iter  130: Obj = 2520.4403, Primal = 0.2391
ADMM Iter  140: Obj = 2626.5523, Primal = 0.2510
ADMM Iter  150: Obj = 2315.8813, Primal = 0.2386
ADMM Iter  160: Obj = 1971.3695, Primal = 0.2238
ADMM Iter  170: Obj = 1313.6004, Primal = 0.1681
ADMM Iter  180: Obj = 741.3968, Primal = 0.1069
ADMM Iter  190: Obj = 694.8092, Primal = 0.109